[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SeyhaLite/ml-foundations/blob/main/docs/notebooks/04_lora_config_lab.ipynb)

# ០៤ · LoRA Config Lab — ពិសោធន៍ប៉ារ៉ាម៉ែត្រ

In [ ]:
# ⚙️ Setup — រត់ cell នេះមុនគេ (Auto-setup for Google Colab)
import os, subprocess, sys

if "google.colab" in sys.modules:
    os.chdir("/content")
    if not os.path.isdir("ml-foundations"):
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/SeyhaLite/ml-foundations.git"], check=True)
    os.chdir("ml-foundations")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)

# ធ្វើឱ្យ qwen_ft import បានភ្លាមៗក្នុង session នេះ (មិនបាច់ restart)
for _cand in ("src", os.path.join("..", "..", "src")):
    _cand = os.path.abspath(_cand)
    if os.path.isdir(_cand) and _cand not in sys.path:
        sys.path.insert(0, _cand)

import qwen_ft
print("qwen-ft", qwen_ft.__version__, "ready ✅")

## LoRA ក្នុងមួយសមីការ

$$W' = W + \frac{\alpha}{r} \cdot BA$$

- $W$ (d×k) — ទម្ងន់ដើម **បង្កក**
- $A$ (r×k), $B$ (d×r) — ម៉ាទ្រីសតូចដែល **ហ្វឹក**
- ប៉ារ៉ាម៉ែត្រ LoRA ក្នុងមួយស្រទាប់ = $r \times (d + k)$

## គណនាចំនួនប៉ារ៉ាម៉ែត្រ — Qwen3.5-4B (វិមាត្រពិតពី config.json)

hidden=2560 · intermediate=9216 · 32 ស្រទាប់ · attention 16Q/4KV heads (head_dim 256)

In [ ]:
hidden, inter, layers = 2560, 9216, 32
q_heads, kv_heads, head_dim = 16, 4, 256
d_q, d_kv = q_heads * head_dim, kv_heads * head_dim   # 4096, 1024

def lora_params(r):
    per_layer = 0
    # attention projections (Gated Attention layers = 1/4 នៃស្រទាប់; DeltaNet មាន projections ផ្ទាល់)
    per_layer += r * (d_q + hidden)      # q_proj
    per_layer += r * (d_kv + hidden)     # k_proj
    per_layer += r * (d_kv + hidden)     # v_proj
    per_layer += r * (hidden + d_q)      # o_proj
    # MLP
    per_layer += r * (inter + hidden)    # gate_proj
    per_layer += r * (inter + hidden)    # up_proj
    per_layer += r * (hidden + inter)    # down_proj
    return per_layer * layers

for r in (8, 16, 32, 64):
    p = lora_params(r)
    print(f"r={r:3d} → {p/1e6:6.1f}M trainable ({p/4e9*100:.2f}% នៃ 4B)")

```{note}
ការគណនានេះប្រហាក់ប្រហែល (មិនរាប់ស្រទាប់ Gated DeltaNet ទាំងអស់លម្អិត) —
ពេលហ្វឹកពិត `peft` បង្ហាញលេខជាក់លាក់។ គោលបំណង៖ ឃើញថា r=16 → ~០.៥% params។
```

## Validation របស់ qwen_ft.config

In [ ]:
from qwen_ft.config import LoRAConfig, ConfigError

for kwargs in [dict(r=16, alpha=32), dict(r=0), dict(dropout=1.5),
               dict(target_modules="everything"), dict(r=16, alpha=8)]:
    cfg = LoRAConfig(**kwargs)
    try:
        cfg.validate()
        status = "✅ valid"
    except ConfigError as e:
        status = f"❌ {e}"
    warns = cfg.warnings()
    print(f"{str(kwargs)[:60]:62s} {status}")
    for w in warns:
        print(f"{'':64s}⚠️  {w}")

## មើល LoRA ពិតលើម៉ូដែល (guarded)

```bash
export QWEN_FT_ALLOW_HEAVY=1   # + pip install "transformers>=5.8,<6" peft
```

In [ ]:
import os
ALLOW_HEAVY = os.environ.get("QWEN_FT_ALLOW_HEAVY", "0") == "1"
if ALLOW_HEAVY:
    # រាប់ trainable % ពិតដោយ peft
    from peft import LoraConfig
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import torch

    MODEL = "Qwen/Qwen3.5-0.8B"
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16)
    lcfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                      target_modules="all-linear", task_type="CAUSAL_LM")
    model.add_adapter(lcfg)
    model.print_trainable_parameters()
else:
    print("⏭️  Heavy demo skipped — set QWEN_FT_ALLOW_HEAVY=1 (requires torch/transformers + model download).")

## លំហាត់

1. គណនា trainable % សម្រាប់ Qwen3.5-9B (hidden 4096-ស្រដៀង; រកវិមាត្រពី model card)
2. ហេតុអ្វី alpha=2r ជាទម្លាប់? សាក r=16/alpha=16 vs 16/32 — តើ scaling ខុសគ្នាយ៉ាងណា?
3. អានជំពូក [៥ · LoRA ពីគោលការណ៍](../chapters/ch05-lora-fundamentals.md)